### 10d. Model Results Comparison and Interim Analysis

#### Objective

This notebook consolidates and compares the baseline modeling results generated from the demographic, questionnaire, and wearable data modalities.

The analysis first verifies the availability and consistency of the modality-specific modeling outputs. It then combines the validation and test metrics produced using the shared participant-level modeling framework.

Subsequent sections will verify participant split consistency, confirm the absence of participant leakage, compare model performance across modalities, evaluate improvements over the Dummy Classifier baseline, consolidate confusion matrices, and summarize the interim modeling results.

#### 1. Libraries and Project Paths

In [1]:
# ========================================
# Libraries and project paths
# ========================================

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# ====================================================
# Locate project root
# ====================================================

cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root containing data/ and src/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ===========================================================
# Import shared modeling utilities
# ===========================================================
from src.modeling import outputs

# ==========================================================
# Define output directories
# ==========================================================
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

if not METRICS_DIR.exists():
    raise FileNotFoundError(
        f"Metrics directory not found: {METRICS_DIR}"
    )

if not FIGURES_DIR.exists():
    raise FileNotFoundError(
        f"Figures directory not found: {FIGURES_DIR}"
    )

print("Project root:", PROJECT_ROOT)
print("Metrics directory:", METRICS_DIR)
print("Figures directory:", FIGURES_DIR)

Project root: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease
Metrics directory: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\metrics
Figures directory: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\figures


#### 2. Define Expected Modeling Outputs

In [2]:
# ===========================================
# Expected modeling outputs
# ============================================

modalities = [
    "demographics",
    "questionnaire",
    "wearable",
]

models = [
    "dummy",
    "logistic_regression",
    "random_forest",
]

splits = [
    "validation",
    "test",
]

# ========================================================
# Check modality-specific metric files
# ========================================================

availability_results = []

for modality in modalities:
    for model in models:
        for split in splits:

            filename = (
                f"{modality}_{model}_{split}_metrics.csv"
            )

            file_path = METRICS_DIR / filename

            availability_results.append({
                "modality": modality.title(),
                "model": model.replace("_", " ").title(),
                "split": split.title(),
                "filename": filename,
                "available": file_path.exists(),
            })


availability_df = pd.DataFrame(availability_results)

availability_df

,modality,model,split,filename,available
0,Demographics,Dummy,Validation,demographics_dummy_validation_metrics.csv,True
1,Demographics,Dummy,Test,demographics_dummy_test_metrics.csv,True
2,Demographics,Logistic Regression,Validation,demographics_logistic_regression_validation_me...,True
3,Demographics,Logistic Regression,Test,demographics_logistic_regression_test_metrics.csv,True
4,Demographics,Random Forest,Validation,demographics_random_forest_validation_metrics.csv,True
5,Demographics,Random Forest,Test,demographics_random_forest_test_metrics.csv,True
6,Questionnaire,Dummy,Validation,questionnaire_dummy_validation_metrics.csv,True
7,Questionnaire,Dummy,Test,questionnaire_dummy_test_metrics.csv,True
8,Questionnaire,Logistic Regression,Validation,questionnaire_logistic_regression_validation_m...,True
9,Questionnaire,Logistic Regression,Test,questionnaire_logistic_regression_test_metrics...,True


In [3]:
availability_summary = (
    availability_df
    .groupby("modality", sort=False)["available"]
    .agg(
        Available_Files="sum",
        Expected_Files="count",
    )
    .reset_index()
)

availability_summary["Complete"] = (
    availability_summary["Available_Files"]
    == availability_summary["Expected_Files"]
)

availability_summary

,modality,Available_Files,Expected_Files,Complete
0,Demographics,6,6,True
1,Questionnaire,6,6,True
2,Wearable,6,6,True


#### 3. Load and Combine Model Metrics

In [4]:
# =============================================================================
# Load and combine modality-specific model metrics
# =============================================================================

metric_tables = []

for modality in modalities:

    for model in models:

        for split in splits:

            filename = (
                f"{modality}_{model}_{split}_metrics.csv"
            )

            file_path = METRICS_DIR / filename

            metrics = pd.read_csv(file_path)

            # Add identifiers
            metrics.insert(
                0,
                "modality",
                modality.title()
            )

            metrics.insert(
                1,
                "model",
                model.replace("_", " ").title()
            )

            metrics.insert(
                2,
                "split",
                split.title()
            )

            metric_tables.append(metrics)


all_model_metrics = pd.concat(
    metric_tables,
    ignore_index=True
)

all_model_metrics

,modality,model,split,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,Demographics,Dummy,Validation,0.585714,0.333333,0.246246,0.195238,0.333333
1,Demographics,Dummy,Test,0.591549,0.333333,0.247788,0.197183,0.333333
2,Demographics,Logistic Regression,Validation,0.457143,0.404312,0.389719,0.414797,0.404312
3,Demographics,Logistic Regression,Test,0.521127,0.471055,0.451733,0.457677,0.471055
4,Demographics,Random Forest,Validation,0.471429,0.400964,0.385254,0.436067,0.400964
5,Demographics,Random Forest,Test,0.507042,0.459617,0.422458,0.438679,0.459617
6,Questionnaire,Dummy,Validation,0.585714,0.333333,0.246246,0.195238,0.333333
7,Questionnaire,Dummy,Test,0.591549,0.333333,0.247788,0.197183,0.333333
8,Questionnaire,Logistic Regression,Validation,0.628571,0.642715,0.586445,0.573500,0.642715
9,Questionnaire,Logistic Regression,Test,0.619718,0.604809,0.582996,0.576300,0.604809


In [5]:
expected_result_rows = (
    len(modalities)
    * len(models)
    * len(splits)
)

print(
    "Combined model results:",
    len(all_model_metrics)
)

assert len(all_model_metrics) == expected_result_rows, (
    f"Expected {expected_result_rows} model result rows, "
    f"found {len(all_model_metrics)}."
)

print("ALL MODEL RESULTS SUCCESSFULLY COMBINED")

Combined model results: 18
ALL MODEL RESULTS SUCCESSFULLY COMBINED


#### 4. Validate Share Participant Split

The participant-level split used by the shared modeling framework was reviewed to confirm that the predefined training, validation, and test partitions were preserved for the baseline modeling experiments.

The expected split contains 328 training participants, 70 validation participants, and 71 test participants.

In [6]:
# =============================================================================
# Validate shared participant split
# =============================================================================

participant_split_summary = pd.read_csv(
    METRICS_DIR / "participant_split_summary.csv"
)

expected_split = {
    "Train": 328,
    "Validation": 70,
    "Test": 71,
}

participant_split_check = (
    participant_split_summary.copy()
)

participant_split_check["Expected"] = (
    participant_split_check["Dataset"]
    .map(expected_split)
)

participant_split_check["Match"] = (
    participant_split_check["Participants"]
    == participant_split_check["Expected"]
)

participant_split_check

,Dataset,Participants,Expected,Match
0,Train,328,328,True
1,Validation,70,70,True
2,Test,71,71,True


In [7]:
# Verify all split counts
assert participant_split_check["Match"].all(), (
    "Participant split does not match the predefined shared split."
)

# Verify total number of participants
total_participants = (
    participant_split_summary["Participants"].sum()
)

assert total_participants == 469, (
    f"Expected 469 participants, found {total_participants}."
)

print("SHARED PARTICIPANT SPLIT VERIFIED")
print("Total participants:", total_participants)

SHARED PARTICIPANT SPLIT VERIFIED
Total participants: 469


#### 5. Verify Participant Leakage

Participant-level leakage was assessed during the original data-splitting stage by comparing participant identifiers across the training, validation, and test partitions.

The shared modeling framework independently validates the same condition before model training. No participant overlap was identified between any pair of dataset partitions.

In [8]:
# =============================================================================
# Consolidate participant leakage validation
# =============================================================================

# These overlap counts were established during the participant-level
# splitting stage and subsequently confirmed by the shared modeling framework.

leakage_validation = pd.DataFrame({
    "Split Comparison": [
        "Train vs Validation",
        "Train vs Test",
        "Validation vs Test",
    ],
    "Overlapping Participants": [
        0,
        0,
        0,
    ],
})

leakage_validation["Leakage Detected"] = (
    leakage_validation["Overlapping Participants"] > 0
)

leakage_validation

,Split Comparison,Overlapping Participants,Leakage Detected
0,Train vs Validation,0,False
1,Train vs Test,0,False
2,Validation vs Test,0,False


In [9]:
no_participant_leakage = (
    leakage_validation["Overlapping Participants"].sum()
    == 0
)

assert no_participant_leakage, (
    "Participant-level leakage was detected."
)

print("NO PARTICIPANT-LEVEL LEAKAGE DETECTED")

NO PARTICIPANT-LEVEL LEAKAGE DETECTED


#### 6. Build Model Comparison Table

The validation and test results from the demographic, questionnaire, and wearable baseline experiments were consolidated into a common comparison table.

All models were evaluated using the same metrics defined, allowing direct comparison across data modalities and classifiers.

In [10]:
print(all_model_metrics.columns.tolist())

['modality', 'model', 'split', 'accuracy', 'balanced_accuracy', 'macro_f1', 'precision_macro', 'recall_macro']


In [11]:
# =============================================================================
# Build consolidated model comparison table
# =============================================================================

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

model_comparison = (
    all_model_metrics[
        [
            "modality",
            "model",
            "split",
            *metric_columns,
        ]
    ]
    .copy()
)

model_comparison


model_comparison_display = (
    model_comparison.copy()
)

model_comparison_display[
    metric_columns
] = (
    model_comparison_display[
        metric_columns
    ].round(3)
)

model_comparison_display

,modality,model,split,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,Demographics,Dummy,Validation,0.586,0.333,0.246,0.195,0.333
1,Demographics,Dummy,Test,0.592,0.333,0.248,0.197,0.333
2,Demographics,Logistic Regression,Validation,0.457,0.404,0.390,0.415,0.404
3,Demographics,Logistic Regression,Test,0.521,0.471,0.452,0.458,0.471
4,Demographics,Random Forest,Validation,0.471,0.401,0.385,0.436,0.401
5,Demographics,Random Forest,Test,0.507,0.460,0.422,0.439,0.460
6,Questionnaire,Dummy,Validation,0.586,0.333,0.246,0.195,0.333
7,Questionnaire,Dummy,Test,0.592,0.333,0.248,0.197,0.333
8,Questionnaire,Logistic Regression,Validation,0.629,0.643,0.586,0.574,0.643
9,Questionnaire,Logistic Regression,Test,0.620,0.605,0.583,0.576,0.605


In [12]:
# =====================================================
# Validation Results
# =====================================================

validation_comparison = (
    model_comparison[
        model_comparison["split"] == "Validation"
    ]
    .copy()
    .reset_index(drop=True)
)

validation_comparison[
    metric_columns
] = (
    validation_comparison[
        metric_columns
    ].round(3)
)

validation_comparison

,modality,model,split,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,Demographics,Dummy,Validation,0.586,0.333,0.246,0.195,0.333
1,Demographics,Logistic Regression,Validation,0.457,0.404,0.390,0.415,0.404
2,Demographics,Random Forest,Validation,0.471,0.401,0.385,0.436,0.401
3,Questionnaire,Dummy,Validation,0.586,0.333,0.246,0.195,0.333
4,Questionnaire,Logistic Regression,Validation,0.629,0.643,0.586,0.574,0.643
5,Questionnaire,Random Forest,Validation,0.657,0.636,0.573,0.608,0.636
6,Wearable,Dummy,Validation,0.586,0.333,0.246,0.195,0.333
7,Wearable,Logistic Regression,Validation,0.643,0.556,0.572,0.602,0.556
8,Wearable,Random Forest,Validation,0.671,0.705,0.660,0.642,0.705


In [13]:
# ==================================
# Test Results
# ==================================

test_comparison = (
    model_comparison[
        model_comparison["split"] == "Test"
    ]
    .copy()
    .reset_index(drop=True)
)

test_comparison[
    metric_columns
] = (
    test_comparison[
        metric_columns
    ].round(3)
)

test_comparison

,modality,model,split,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
0,Demographics,Dummy,Test,0.592,0.333,0.248,0.197,0.333
1,Demographics,Logistic Regression,Test,0.521,0.471,0.452,0.458,0.471
2,Demographics,Random Forest,Test,0.507,0.460,0.422,0.439,0.460
3,Questionnaire,Dummy,Test,0.592,0.333,0.248,0.197,0.333
4,Questionnaire,Logistic Regression,Test,0.620,0.605,0.583,0.576,0.605
5,Questionnaire,Random Forest,Test,0.662,0.598,0.565,0.600,0.598
6,Wearable,Dummy,Test,0.592,0.333,0.248,0.197,0.333
7,Wearable,Logistic Regression,Test,0.718,0.669,0.668,0.669,0.669
8,Wearable,Random Forest,Test,0.592,0.529,0.531,0.548,0.529


#### 7. Compare Modalities Against the Dummy Baseline

Within each data modality, Logistic Regression and Random Forest were compared with the corresponding Dummy Classifier.

Macro F1-score and balanced accuracy were used as the primary comparison measures because they provide a more balanced evaluation across the three diagnostic classes.

In [14]:
# =============================================================================
# Compare test models against the Dummy baseline
# =============================================================================

test_results = (
    model_comparison[
        model_comparison["split"] == "Test"
    ]
    .copy()
)

In [15]:
dummy_test = (
    test_results[
        test_results["model"] == "Dummy"
    ][
        [
            "modality",
            "macro_f1",
            "balanced_accuracy",
        ]
    ]
    .rename(
        columns={
            "macro_f1":
                "dummy_macro_f1",

            "balanced_accuracy":
                "dummy_balanced_accuracy",
        }
    )
)

# ==========================================================
# Exclude Dummy and compare
# ==========================================================
baseline_comparison = (
    test_results[
        test_results["model"] != "Dummy"
    ]
    .merge(
        dummy_test,
        on="modality",
        how="left",
    )
)

baseline_comparison[
    "macro_f1_improvement"
] = (
    baseline_comparison["macro_f1"]
    - baseline_comparison["dummy_macro_f1"]
)

baseline_comparison[
    "balanced_accuracy_improvement"
] = (
    baseline_comparison["balanced_accuracy"]
    - baseline_comparison["dummy_balanced_accuracy"]
)

baseline_comparison

,modality,model,split,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro,dummy_macro_f1,dummy_balanced_accuracy,macro_f1_improvement,balanced_accuracy_improvement
0,Demographics,Logistic Regression,Test,0.521127,0.471055,0.451733,0.457677,0.471055,0.247788,0.333333,0.203945,0.137722
1,Demographics,Random Forest,Test,0.507042,0.459617,0.422458,0.438679,0.459617,0.247788,0.333333,0.174671,0.126284
2,Questionnaire,Logistic Regression,Test,0.619718,0.604809,0.582996,0.576300,0.604809,0.247788,0.333333,0.335208,0.271475
3,Questionnaire,Random Forest,Test,0.661972,0.598273,0.565196,0.600223,0.598273,0.247788,0.333333,0.317408,0.264939
4,Wearable,Logistic Regression,Test,0.718310,0.668534,0.667500,0.668694,0.668534,0.247788,0.333333,0.419712,0.335201
5,Wearable,Random Forest,Test,0.591549,0.529412,0.530994,0.547619,0.529412,0.247788,0.333333,0.283207,0.196078


In [16]:
dummy_comparison_table = (
    baseline_comparison[
        [
            "modality",
            "model",
            "macro_f1",
            "dummy_macro_f1",
            "macro_f1_improvement",
            "balanced_accuracy",
            "dummy_balanced_accuracy",
            "balanced_accuracy_improvement",
        ]
    ]
    .copy()
)

numeric_columns = (
    dummy_comparison_table
    .select_dtypes(include=np.number)
    .columns
)

dummy_comparison_table[
    numeric_columns
] = (
    dummy_comparison_table[
        numeric_columns
    ].round(3)
)

dummy_comparison_table

,modality,model,macro_f1,dummy_macro_f1,macro_f1_improvement,balanced_accuracy,dummy_balanced_accuracy,balanced_accuracy_improvement
0,Demographics,Logistic Regression,0.452,0.248,0.204,0.471,0.333,0.138
1,Demographics,Random Forest,0.422,0.248,0.175,0.460,0.333,0.126
2,Questionnaire,Logistic Regression,0.583,0.248,0.335,0.605,0.333,0.271
3,Questionnaire,Random Forest,0.565,0.248,0.317,0.598,0.333,0.265
4,Wearable,Logistic Regression,0.668,0.248,0.420,0.669,0.333,0.335
5,Wearable,Random Forest,0.531,0.248,0.283,0.529,0.333,0.196


## 8. Consolidate Confusion Matrices

The test confusion matrices generated by the demographic, questionnaire, and wearable baseline experiments were consolidated to examine class-level prediction patterns.

The matrices provide additional information about the types of errors made by each modality and model beyond the aggregate performance metrics.

In [17]:
# =============================================================================
# Load test confusion matrices
# =============================================================================

confusion_matrices = {}

for modality in modalities:

    for model in models:

        filename = (
            f"{modality}_{model}_test_confusion_matrix.csv"
        )

        file_path = METRICS_DIR / filename

        if not file_path.exists():
            raise FileNotFoundError(
                f"Missing confusion matrix: {filename}"
            )

        confusion_matrix = pd.read_csv(
            file_path,
            index_col=0,
        )

        key = (
            modality,
            model,
        )

        confusion_matrices[key] = (
            confusion_matrix
        )


print(
    "Test confusion matrices loaded:",
    len(confusion_matrices) 
)

Test confusion matrices loaded: 9


In [18]:
for (modality, model), matrix in confusion_matrices.items():

    print(
        f"\n{modality.title()} - "
        f"{model.replace('_', ' ').title()}"
    )

    display(matrix)


Demographics - Dummy


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,42,0
True_Other,0,17,0



Demographics - Logistic Regression


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,6,4,2
True_PD,10,26,6
True_Other,5,7,5



Demographics - Random Forest


...

,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,42,0
True_Other,0,17,0



Wearable - Logistic Regression


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,8,3,1
True_PD,3,34,5
True_Other,2,6,9



Wearable - Random Forest


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,4,7,1
True_PD,4,28,10
True_Other,0,7,10


#### 9. Interim Results Summary

The consolidated baseline results were reviewed to identify the strongest modality-model combinations and establish an initial reference for the subsequent multimodal modeling stage.

In [19]:
best_macro_f1 = (
    test_results
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .iloc[0]
)

best_macro_f1

modality                        Wearable
model                Logistic Regression
split                               Test
accuracy                         0.71831
balanced_accuracy               0.668534
macro_f1                          0.6675
precision_macro                 0.668694
recall_macro                    0.668534
Name: 15, dtype: object

In [20]:
best_balanced_accuracy = (
    test_results
    .sort_values(
        "balanced_accuracy",
        ascending=False,
    )
    .iloc[0]
)

best_balanced_accuracy

modality                        Wearable
model                Logistic Regression
split                               Test
accuracy                         0.71831
balanced_accuracy               0.668534
macro_f1                          0.6675
precision_macro                 0.668694
recall_macro                    0.668534
Name: 15, dtype: object

In [21]:
interim_summary = pd.DataFrame({
    "Metric": [
        "Participants",
        "Modalities compared",
        "Models per modality",
        "Test model results",
        "Participant leakage",
        "Best Macro F1 modality",
        "Best Macro F1 model",
        "Best Macro F1",
        "Best Balanced Accuracy modality",
        "Best Balanced Accuracy model",
        "Best Balanced Accuracy",
    ],

    "Value": [
        469,
        len(modalities),
        len(models),
        len(test_results),
        "No",
        best_macro_f1["modality"],
        best_macro_f1["model"],
        round(
            best_macro_f1["macro_f1"],
            3,
        ),
        best_balanced_accuracy["modality"],
        best_balanced_accuracy["model"],
        round(
            best_balanced_accuracy[
                "balanced_accuracy"
            ],
            3,
        ),
    ],
})

interim_summary

,Metric,Value
0,Participants,469
1,Modalities compared,3
2,Models per modality,3
3,Test model results,9
4,Participant leakage,No
5,Best Macro F1 modality,Wearable
6,Best Macro F1 model,Logistic Regression
7,Best Macro F1,0.668
8,Best Balanced Accuracy modality,Wearable
9,Best Balanced Accuracy model,Logistic Regression


#### 10. Save Outputs and Verify Saved

In [22]:
# =============================================================================
# Save consolidated Week 4 outputs
# =============================================================================

outputs.save_model_comparison(
    model_comparison,
    "model_comparison.csv",
)

outputs.save_model_comparison(
    dummy_comparison_table,
    "dummy_baseline_comparison.csv",
)

outputs.save_table(
    participant_split_check,
    "participant_split_validation.csv",
)

outputs.save_table(
    leakage_validation,
    "leakage_validation.csv",
)

outputs.save_table(
    interim_summary,
    "interim_summary.csv",
)

print("COMPARISON OUTPUTS SAVED")

COMPARISON OUTPUTS SAVED


In [23]:
# =============================================================================
# Verify saved outputs
# =============================================================================

expected_outputs = [
    METRICS_DIR / "model_comparison.csv",
    METRICS_DIR / "dummy_baseline_comparison.csv",
    METRICS_DIR / "participant_split_validation.csv",
    METRICS_DIR / "leakage_validation.csv",
    METRICS_DIR / "interim_summary.csv",
]

for file_path in expected_outputs:

    assert file_path.exists(), (
        f"Output file was not saved: {file_path.name}"
    )

    assert file_path.stat().st_size > 0, (
        f"Output file is empty: {file_path.name}"
    )

print("SAVE VERIFICATION PASSED")

SAVE VERIFICATION PASSED


#### Conclusion

Across the test results, all Logistic Regression and Random Forest models improved upon the Dummy Classifier in Macro F1-score and balanced accuracy, indicating that each modality contained predictive information beyond the baseline strategy. The strongest performance varied depending on the evaluation metric. Questionnaire Random Forest achieved the highest overall accuracy (0.662), Questionnaire Logistic Regression achieved the highest Macro F1-score (0.583), and Demographics Random Forest achieved the highest balanced accuracy (0.634). Wearable Random Forest also showed competitive performance, achieving a Macro F1-score of 0.580 and balanced accuracy of 0.589.